
## 📦 Install Dependencies

In [1]:
!pip install -q accelerate bitsandbytes datasets huggingface_hub peft scikit-learn transformers trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 45.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 530.9/530.9 kB 50.2 MB/s eta 0:00:00


## 📚 Libraries

In [27]:
from collections import Counter
from datasets import load_dataset
from google.colab import userdata
from huggingface_hub import login

import os
import random
import numpy as np
import torch
from torch.nn import CrossEntropyLoss

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
)

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    default_data_collator,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
)

from peft import LoraConfig, get_peft_model, PeftModel

🔕 Disable Weights & Biases

In [3]:
os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_MODE"] = "offline"

## 🔐 Login to Hugging Face Hub

In [4]:
hf_token = os.environ.get('HF_Token') or userdata.get('HF_Token')

if hf_token:
    login(token=hf_token)
    print("HuggingFace login successful.")
else:
    print("HuggingFace token not found. Please set the HF_TOKEN environment variable or store it in Colab secrets.")

HuggingFace login successful.


## 📥 Load dair-ai/emotion Dataset

In [5]:
dataset = load_dataset("dair-ai/emotion")

emotion_map = {
    0: "sadness", 1: "joy", 2: "love",
    3: "anger",   4: "fear", 5: "surprise"
}

def convert_label(ex):
    ex["emotion"] = emotion_map[int(ex["label"])]
    return ex

dataset["train"] = dataset["train"].map(convert_label)
dataset["test"]  = dataset["test"].map(convert_label)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

split/train-00000-of-00001.parquet:   0%|          | 0.00/1.03M [00:00<?, ?B/s]

split/validation-00000-of-00001.parquet:   0%|          | 0.00/127k [00:00<?, ?B/s]

split/test-00000-of-00001.parquet:   0%|          | 0.00/129k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/16000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/16000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

⚖️ Inspect Dataset Balance

In [6]:
label_counts = Counter(dataset["train"]["label"])
print("\nLabel counts in train:", label_counts)


Label counts in train: Counter({1: 5362, 0: 4666, 3: 2159, 4: 1937, 2: 1304, 5: 572})


⚖️ Class Weights

In [7]:
counts = torch.tensor([label_counts[i] for i in range(6)], dtype=torch.float)
weights = 1.0 / counts
# I keep mean=1 so the overall loss scale stays reasonable
weights = weights / weights.mean()

print("\nClass weights:", weights)


Class weights: tensor([0.3301, 0.2873, 1.1812, 0.7134, 0.7952, 2.6928])


The dataset is unbalanced.  And so I add class weights (loss weights) so that more rare classes have more weight

🧠 Options & Global Constants

In [8]:
OPTIONS = ["sadness", "joy", "love", "anger", "fear", "surprise"]
label2id = {label: i for i, label in enumerate(OPTIONS)}

## 🧩 Create Prompts from Tabular Data



In [10]:
def row_to_prompt(example):
    options_text = "\n".join(f"- {opt}" for opt in OPTIONS)
    prompt = (
        "You are an expert in emotion classification.\n\n"
        f"Text:\n{example['text']}\n\n"
        f"Options:\n{options_text}\n\n"
        "Answer with exactly one option from the list."
    )
    return {
        "prompt": prompt,
        "label_id": label2id[example["emotion"]],
    }

train_ds = dataset["train"].map(row_to_prompt)
test_ds  = dataset["test"].map(row_to_prompt)

print("\nExample prompt row:\n")
print(train_ds[0]["prompt"])
print("label_id:", train_ds[0]["label_id"])

Map:   0%|          | 0/16000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]


Example prompt row:

You are an expert in emotion classification.

Text:
i didnt feel humiliated

Options:
- sadness
- joy
- love
- anger
- fear
- surprise

Answer with exactly one option from the list.
label_id: 0


🧪 Reproducibility

In [11]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

Hardware Configuration

In [12]:
use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
compute_dtype = torch.bfloat16 if use_bf16 else torch.float16
print(f"\nUsing bf16? {use_bf16}  | compute_dtype={compute_dtype}")


Using bf16? True  | compute_dtype=torch.bfloat16


🧠 Shared Tokenizer

In [13]:
MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"

tok = AutoTokenizer.from_pretrained(
    MODEL_ID,
    token=hf_token,
    use_fast=True,
)

# pad token setup (important for batching)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

# We'll classify by looking only at these token IDs
# NOTE: This is still a "last-token" trick — it's simple and fast, but not perfect.
answer_token_ids = [
    tok(opt, add_special_tokens=False).input_ids[-1]
    for opt in OPTIONS
]
answer_token_ids_tensor = torch.tensor(answer_token_ids, dtype=torch.long)

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

🔢 Tokenize for Training

In [14]:
MAX_LEN = 512

def tokenize_fn(batch):
    enc = tok(
        batch["prompt"],
        truncation=True,
        padding="max_length",   # simple + predictable
        max_length=MAX_LEN,
    )
    enc["labels"] = batch["label_id"]  # class index 0..5
    return enc

train_encoded = train_ds.map(
    tokenize_fn,
    batched=True,
    remove_columns=train_ds.column_names,
)
test_encoded = test_ds.map(
    tokenize_fn,
    batched=True,
    remove_columns=test_ds.column_names,
)

train_encoded.set_format(type="torch")
test_encoded.set_format(type="torch")

Map:   0%|          | 0/16000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

🧠 Baseline Model (Unfine-tuned)

In [15]:
baseline_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    torch_dtype=compute_dtype,
    token=hf_token,
)
print("\nLoaded baseline_model (unfine-tuned).")


`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]


Loaded baseline_model (unfine-tuned).


📊 Shared Evaluation Function

In [17]:
def evaluate_model(model, ds_with_prompts, name: str, n_preview: int = 500):
    """
    Evaluate using the same "next-token over options" trick:
      - Run model on prompt
      - Grab logits at the final position
      - Compare logits for the 6 option tokens
    """
    model.eval()
    N = min(n_preview, len(ds_with_prompts))

    y_true, y_pred = [], []

    for i in range(N):
        ex = ds_with_prompts[i]
        prompt = ex["prompt"]
        label_id = ex["label_id"]

        inputs = tok(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=MAX_LEN,
        ).to(model.device)

        with torch.no_grad():
            outputs = model(**inputs)

            # last non-pad token index (safe even if we pad later)
            attn = inputs["attention_mask"]  # [1, T]
            last_idx = int(attn.sum(dim=1).item() - 1)

            logits_full = outputs.logits[:, last_idx, :]                  # [1, vocab]
            option_logits = logits_full[:, answer_token_ids_tensor.to(model.device)]  # [1, 6]
            pred_class = option_logits.argmax(dim=-1).item()

        y_true.append(label_id)
        y_pred.append(pred_class)

    acc = accuracy_score(y_true, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="macro",
        zero_division=0,
    )

    print(f"\n====== {name} (N={N}) ======")
    print(f"Acc={acc:.3f}  Prec={prec:.3f}  Rec={rec:.3f}  F1={f1:.3f}\n")
    print("Classification report:")
    print(classification_report(y_true, y_pred, target_names=OPTIONS, zero_division=0))

    cm = confusion_matrix(y_true, y_pred)
    print("\nConfusion matrix (rows=true, cols=pred):")
    print("   " + "  ".join(OPTIONS))
    for r, row in enumerate(cm):
        print(f"{OPTIONS[r]:8s} {row}")

    return y_true, y_pred, cm


📊 Baseline Evaluation

In [18]:
baseline_evaluation = evaluate_model(baseline_model, test_ds, name="Baseline (unfine-tuned)")


====== Baseline (unfine-tuned) (N=500) ======
Acc=0.284  Prec=0.196  Rec=0.279  F1=0.182

Classification report:
              precision    recall  f1-score   support

     sadness       0.00      0.00      0.00       148
         joy       0.80      0.38      0.51       151
        love       0.17      0.44      0.24        39
       anger       0.21      0.86      0.33        79
        fear       0.00      0.00      0.00        70
    surprise       0.00      0.00      0.00        13

    accuracy                           0.28       500
   macro avg       0.20      0.28      0.18       500
weighted avg       0.29      0.28      0.23       500


Confusion matrix (rows=true, cols=pred):
   sadness  joy  love  anger  fear  surprise
sadness  [  0   4  24 120   0   0]
joy      [ 0 57 38 56  0  0]
love     [ 0  7 17 15  0  0]
anger    [ 0  2  9 68  0  0]
fear     [ 0  1  8 61  0  0]
surprise [0 0 5 8 0 0]


⚙️ QLoRA Base Model (4-bit) for Training

In [19]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
)

sft_base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    token=hf_token,
)
print("\nLoaded sft_base_model (4-bit, QLoRA-ready).")

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

sft_model = get_peft_model(sft_base_model, lora_config)
print("Wrapped sft_base_model with LoRA → sft_model.")

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]


Loaded sft_base_model (4-bit, QLoRA-ready).
Wrapped sft_base_model with LoRA → sft_model.


🛠️ Custom Trainer with loss_fn

In [20]:
class WeightedLossTrainer(Trainer):
    """
    Trainer that:
    - Uses class-weighted CrossEntropyLoss
    - Scores only logits over the 6 emotion option tokens
    - Uses attention_mask to find the real last token (with max_length padding)
    """
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs["labels"]  # [B]
        outputs = model(**{k: v for k, v in inputs.items() if k != "labels"})

        logits_full = outputs.logits  # [B, T, vocab]

        # Find last non-padding token per row
        attn = inputs["attention_mask"]  # [B, T]
        last_idx = attn.sum(dim=1) - 1   # [B]
        batch_idx = torch.arange(attn.size(0), device=attn.device)

        last_logits = logits_full[batch_idx, last_idx, :]  # [B, vocab]
        option_logits = last_logits[:, answer_token_ids_tensor.to(last_logits.device)]  # [B, 6]

        weighted_ce = CrossEntropyLoss(weight=weights.to(option_logits.device))
        loss = weighted_ce(option_logits, labels.to(option_logits.device))

        if return_outputs:
            return loss, outputs
        return loss

🎯 Training Arguments

In [23]:
training_args = TrainingArguments(
    output_dir="emotion-llama-qlora",
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=1e-4,
    num_train_epochs=1,
    bf16=True,
    logging_steps=50,
    eval_steps=200,
    save_steps=200,
    save_total_limit=2,
    report_to="none",   # disable wandb etc.
)

🚀 Train

In [29]:
trainer = WeightedLossTrainer(
    model=sft_model,
    args=training_args,
    train_dataset=train_encoded,
    eval_dataset=test_encoded,
    data_collator=default_data_collator,
)

🚀 QLoRA Fine-Tuning

In [30]:
trainer.train()

Step,Training Loss
50,21.884014
100,16.668910
150,12.781293
200,10.558376
250,9.159428
300,7.694036
350,7.313467
400,6.328127
450,5.982896
500,5.145721


TrainOutput(global_step=1000, training_loss=7.275177047729493, metrics={'train_runtime': 4262.814, 'train_samples_per_second': 3.753, 'train_steps_per_second': 0.235, 'total_flos': 3.69217064927232e+17, 'train_loss': 7.275177047729493, 'epoch': 1.0})

💾 Save LoRA

In [31]:
adapter_dir = "emotion-llama-qlora"
sft_model.save_pretrained(adapter_dir)
tok.save_pretrained(adapter_dir)
print(f"\nSaved LoRA adapter + tokenizer to: {adapter_dir}")

sft_model.push_to_hub("david125tran/emotion-llama-qlora")
tok.push_to_hub("david125tran/emotion-llama-qlora")


Saved LoRA adapter + tokenizer to: emotion-llama-qlora


README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   4%|4         | 1.12MB / 27.3MB            

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpucsch3wv/tokenizer.json: 100%|##########| 17.2MB / 17.2MB            

CommitInfo(commit_url='https://huggingface.co/david125tran/emotion-llama-qlora/commit/b78962f29404f6c2286865807d19e37f840b09ac', commit_message='Upload tokenizer', commit_description='', oid='b78962f29404f6c2286865807d19e37f840b09ac', pr_url=None, repo_url=RepoUrl('https://huggingface.co/david125tran/emotion-llama-qlora', endpoint='https://huggingface.co', repo_type='model', repo_id='david125tran/emotion-llama-qlora'), pr_revision=None, pr_num=None)

🔄 Reload Fine-Tuned Model for Evaluation

In [32]:
reload_base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,   # use same 4-bit config
    device_map="auto",
    token=hf_token,
)

sft_model_reloaded = PeftModel.from_pretrained(
    reload_base,
    "david125tran/emotion-llama-qlora",
)
print("\nReloaded sft_model_reloaded from Hub.")

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

adapter_config.json:   0%|          | 0.00/986 [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/27.3M [00:00<?, ?B/s]


Reloaded sft_model_reloaded from Hub.


📊 Evaluate Fine-Tuned

In [33]:
ft_evaulated = evaluate_model(sft_model_reloaded, test_ds, name="Supervised Fine-Tuning Model")


====== Supervised Fine-Tuning Model (N=500) ======
Acc=0.928  Prec=0.883  Rec=0.906  F1=0.893

Classification report:
              precision    recall  f1-score   support

     sadness       0.96      0.95      0.96       148
         joy       0.95      0.93      0.94       151
        love       0.79      0.85      0.81        39
       anger       0.94      0.95      0.94        79
        fear       0.93      0.91      0.92        70
    surprise       0.73      0.85      0.79        13

    accuracy                           0.93       500
   macro avg       0.88      0.91      0.89       500
weighted avg       0.93      0.93      0.93       500


Confusion matrix (rows=true, cols=pred):
   sadness  joy  love  anger  fear  surprise
sadness  [141   1   0   3   2   1]
joy      [  0 140   9   0   0   2]
love     [ 0  6 33  0  0  0]
anger    [ 3  0  0 75  1  0]
fear     [ 3  0  0  2 64  1]
surprise [ 0  0  0  0  2 11]
